In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
import os
import sys
import subprocess

In [ ]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

In [ ]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

In [ ]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])


def find_model_path(model_name: str = 'gpt-oss-120b') -> str:
    """Auto-discover model path from /kaggle/input/."""
    import glob

    candidates = [
        # model_sources mount pattern (new Kaggle)
        f'/kaggle/input/models/danielhanchen/{model_name}/transformers/default/1',
        f'/kaggle/input/models/openai/{model_name}/transformers/default/1',
        # dataset-style mount pattern (old Kaggle / some configs)
        f'/kaggle/input/{model_name}/transformers/default/1',
        f'/kaggle/input/{model_name}',
    ]

    for path in candidates:
        if os.path.isfile(os.path.join(path, 'config.json')):
            print(f'Model found at: {path}')
            return path

    # Fallback: search for any config.json with model weights
    for config_path in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        parent = os.path.dirname(config_path)
        if any(glob.glob(os.path.join(parent, f'*.{ext}')) for ext in ['safetensors', 'bin']):
            print(f'Model found via search: {parent}')
            return parent

    # Debug dump
    print('ERROR: Model not found. /kaggle/input/ contents:')
    for root, dirs, files in os.walk('/kaggle/input/', topdown=True):
        depth = root.replace('/kaggle/input/', '').count(os.sep)
        if depth < 5:
            print(f'{"  " * depth}{os.path.basename(root)}/ ({len(files)} files)')
        if depth >= 5:
            dirs[:] = []

    raise RuntimeError(f'Model "{model_name}" not found under /kaggle/input/')


MODEL_PATH = find_model_path()
print(f'\nUsing model path: {MODEL_PATH}')

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [ ]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [ ]:
class CFG:
    
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'
        
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
        
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.\n\n'
        
        '# Efficiency:\n'
        'If the problem has an obvious, immediate answer (e.g. direct computation, '
        'well-known identity, or trivial formula application), state the answer '
        'directly with brief justification. Do not use Python for problems you can '
        'solve in 3 lines of reasoning.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'
        
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'
        
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )
    
    preference_prompt = (
        'You have access to `math`, `numpy`, and `sympy` for:\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'
        
        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds\n'
        '- For very large exponents (e.g. a^(n!)), use pow(base, exp, mod) or analytical methods — never materialize the full number\n\n'
        
        '# Code Robustness Rules:\n'
        '- Each code cell must be SELF-CONTAINED: re-import and re-define everything you need. '
        'If a previous cell errored, its definitions are LOST.\n'
        '- Avoid sympy expressions that grow beyond a few hundred characters. '
        'If an expression is exploding, switch to numerical methods (numpy) or modular arithmetic.\n'
        '- For geometry: prefer numpy coordinate models with brute-force verification '
        'over pure symbolic or direction-counting approaches.\n'
        '- For combinatorics counting: always verify no duplicates — check injectivity explicitly.\n'
        '- If 5+ code cells without progress, STOP and restart with a different approach.\n'
        '- Never enumerate more than 10^6 cases without checking feasibility first.\n'
        '- When using Python, prefer converting sympy expressions to int() before passing to Python builtins like pow(), min(), max().\n'
        '- Common import: from sympy.ntheory.modular import crt (note: sympy.crt does not exist).\n'
        '- If a code cell fails, re-import necessary libraries in the next cell.'
    )
    
    served_model_name = 'gpt-oss'
    model_path = MODEL_PATH
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 30
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    early_stop = 5
    attempts = 16
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 0.5
    temp_schedule = [0.1, 0.3, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.7, 0.7, 0.7, 0.7, 0.9]
    min_p = 0.02

In [ ]:
set_seed(CFG.seed)

In [ ]:
class AIMO3Template:

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [ ]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import functools\n'
            'import fractions\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import functools\n'
            'import fractions\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [ ]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

In [ ]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
    
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
    
        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._start_gpu_monitor()
        self._initialize_kernels()
    
        self.notebook_start_time = time.time()
        self.problems_remaining = 50
    
    def _preload_model_weights(self) -> None:
    
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
    
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
    
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:
    
        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--disable-log-stats', 
            '--enable-prefix-caching'
        ]
    
        self.log_file = open('vllm_server.log', 'w')
    
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )
    
    def _wait_for_server(self):
    
        print('Waiting for vLLM server...')
        start_time = time.time()
    
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
    
                return
    
            except Exception:
                time.sleep(1)
    
        raise RuntimeError('Server failed to start (timeout).\n')
    

    def _start_gpu_monitor(self):
        """Background thread polling vLLM /metrics every 5s. Logs to gpu_metrics.log."""
        import requests as _req

        self._gpu_log = open('gpu_metrics.log', 'w', buffering=1)
        self._gpu_log.write('timestamp,elapsed_s,running,waiting,cache_pct,preemptions\n')
        self._monitor_stop = threading.Event()
        self._peak_running = 0
        self._peak_waiting = 0
        self._peak_cache = 0.0
        self._total_samples = 0
        self._sum_running = 0
        self._sum_waiting = 0
        self._wait_samples = 0  # samples where waiting > 0

        def _poll():
            t0 = time.time()
            while not self._monitor_stop.is_set():
                try:
                    resp = _req.get(f'{self.base_url.replace("/v1","")}/metrics', timeout=3)
                    lines = resp.text.split('\n')
                    running = waiting = preemptions = 0
                    cache_pct = 0.0
                    for line in lines:
                        if line.startswith('vllm:num_requests_running{'):
                            running = int(float(line.split()[-1]))
                        elif line.startswith('vllm:num_requests_waiting{'):
                            waiting = int(float(line.split()[-1]))
                        elif line.startswith('vllm:gpu_cache_usage_perc{'):
                            cache_pct = float(line.split()[-1])
                        elif line.startswith('vllm:num_preemptions_total'):
                            preemptions = int(float(line.split()[-1]))

                    elapsed = time.time() - t0
                    self._gpu_log.write(f'{time.time():.1f},{elapsed:.1f},{running},{waiting},{cache_pct:.4f},{preemptions}\n')

                    self._peak_running = max(self._peak_running, running)
                    self._peak_waiting = max(self._peak_waiting, waiting)
                    self._peak_cache = max(self._peak_cache, cache_pct)
                    self._total_samples += 1
                    self._sum_running += running
                    self._sum_waiting += waiting
                    if waiting > 0:
                        self._wait_samples += 1
                except Exception:
                    pass
                self._monitor_stop.wait(5)

        t = threading.Thread(target=_poll, daemon=True)
        t.start()

    def print_gpu_summary(self):
        """Print GPU utilization summary — call after all problems solved."""
        self._monitor_stop.set()
        self._gpu_log.close()
        n = self._total_samples or 1
        print(f'\n{"="*70}')
        print(f'  GPU METRICS SUMMARY ({n} samples @ 5s intervals)')
        print(f'{"="*70}')
        print(f'  Peak concurrent running:  {self._peak_running}')
        print(f'  Peak concurrent waiting:  {self._peak_waiting}')
        print(f'  Peak KV cache usage:      {self._peak_cache:.1%}')
        print(f'  Avg concurrent running:   {self._sum_running/n:.1f}')
        print(f'  Avg concurrent waiting:   {self._sum_waiting/n:.1f}')
        print(f'  Samples with waiting > 0: {self._wait_samples}/{n} ({100*self._wait_samples/n:.1f}%)')
        print(f'{"="*70}')
        if self._peak_waiting == 0:
            print(f'  --> No queuing observed. Could increase attempts further.')
        elif self._wait_samples / n < 0.1:
            print(f'  --> Rare queuing (<10% of samples). Current config is near-optimal.')
        else:
            print(f'  --> Significant queuing. Consider reducing attempts or early_stop.')
        print()

    def _initialize_kernels(self) -> None:
    
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
    
        self.sandbox_pool = queue.Queue()
    
        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
    
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
    
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _scan_for_answer(self, text: str) -> int | None:
        
        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
                
        pattern = r'final\s+answer\s+is\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
    
        # Fallback: "the answer is X", "answer is X", "answer is: X"
        pattern = r'(?:the\s+)?answer\s+is\s*:?\s*\*?\*?\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
    
        # Fallback: "answer: X" or "Answer = X"
        pattern = r'answer\s*[:=]\s*\*?\*?\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
    
        return None
    
    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
    
        if not logprobs_buffer:
            return float('inf')
    
        total_entropy = 0.0
        token_count = 0
    
        for top_logprobs_dict in logprobs_buffer:
            
            if not isinstance(top_logprobs_dict, dict):
                continue
            
            if not top_logprobs_dict:
                continue
            
            token_entropy = 0.0
            
            for token_str, log_prob in top_logprobs_dict.items():
                prob = math.exp(log_prob)
                
                if prob > 0:
                    token_entropy -= prob * math.log2(prob)
            
            total_entropy += token_entropy
            token_count += 1
    
        if token_count == 0:
            return float('inf')
    
        return total_entropy / token_count
    
    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float,
        temperature: float = None
    ) -> dict:
    
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Answer': None, 
                'Temperature': temperature,
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0, 
                'Entropy': float('inf'),
                'Conversation': [],
                'Time': 0
            }
    
        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        
        logprobs_buffer = []
        conversation_log = []
        turn_num = 0
        attempt_start = time.time()
    
        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))
    
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
    
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )
    
            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                problem, 
                local_tool.tool_config
            )
    
            conversation = Conversation.from_messages(messages)
    
            for _ in range(self.cfg.turns):
                turn_num += 1
                if stop_event.is_set() or time.time() > deadline:
                    break
    
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
    
                if max_tokens < self.cfg.buffer_tokens:
                    break
    
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name, 
                    temperature=(temperature if temperature is not None else self.cfg.temperature), 
                    logprobs=self.cfg.top_logprobs, 
                    max_tokens=max_tokens, 
                    prompt=prompt_ids, 
                    seed=attempt_seed, 
                    stream=True, 
                    extra_body={
                        'min_p': self.cfg.min_p, 
                        'stop_token_ids': self.stop_token_ids, 
                        'return_token_ids': True
                    }
                )
    
                try:
                    token_buffer = []
                    text_chunks = []
    
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
    
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
    
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)
                            
                            chunk_logprobs = chunk.choices[0].logprobs
                            
                            if chunk_logprobs is not None:
                                if chunk_logprobs.top_logprobs:
                                    logprobs_buffer.extend(chunk_logprobs.top_logprobs)
    
                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)
    
                            if answer is not None:
                                final_answer = answer
                                break
    
                finally:
                    stream.close()
    
                turn_text = ''.join(text_chunks)
                if turn_text.strip():
                    conversation_log.append({
                        'turn': turn_num,
                        'type': 'reasoning',
                        'text': turn_text
                    })
    
                if final_answer is not None:
                    break
    
                if not token_buffer:
                    break
    
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]
    
                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self._scan_for_answer(answer_text)
                    break
    
                if last_message.recipient == 'python':
                    python_calls += 1
                    raw_code = last_message.content[0].text
                    tool_responses = local_tool.process_sync_plus(last_message)
    
                    response_text = tool_responses[0].content[0].text
    
                    conversation_log.append({
                        'turn': turn_num,
                        'type': 'code_call',
                        'code': raw_code,
                        'output': response_text,
                        'error': response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text
                    })
    
                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                        python_errors += 1
    
                    conversation.messages.extend(tool_responses)
    
        except Exception as exc:
            python_errors += 1
    
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
    
        mean_entropy = self._compute_mean_entropy(logprobs_buffer)
    
        attempt_elapsed = time.time() - attempt_start
    
        return {
            'Attempt': attempt_index + 1, 
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Entropy': mean_entropy, 
            'Temperature': temperature,
            'Answer': final_answer,
            'Conversation': conversation_log,
            'Time': round(attempt_elapsed, 1)
        }
    
    def _select_answer(self, detailed_results: list) -> int:

        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']
            
            if answer is not None:
                weight = 1.0 / max(entropy, 1e-9)
                
                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer, 
                'votes': answer_votes[answer], 
                'score': total_weight
            })

        # Deterministic tie-breaking: score (desc), then votes (desc), then answer value (desc)
        scored_answers.sort(key=lambda x: (x['score'], x['votes'], x['answer']), reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((
                item['answer'], 
                item['votes'], 
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data, 
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)
        
        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        final_answer = scored_answers[0]['answer']    
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer
    
    def solve_problem(self, problem: str) -> int:
    
        print(f'\nProblem: {problem}\n')
        
        user_input = f'{problem} {self.cfg.preference_prompt}'
    
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
    
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
    
        deadline = time.time() + budget
    
        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')
    
        tasks = []
    
        for attempt_index in range(self.cfg.attempts):
            if hasattr(self.cfg, 'temp_schedule') and self.cfg.temp_schedule and attempt_index < len(self.cfg.temp_schedule):
                temp = self.cfg.temp_schedule[attempt_index]
            else:
                temp = self.cfg.temperature
            tasks.append((self.cfg.system_prompt, attempt_index, temp))
    
        detailed_results = []
        valid_answers = []
    
        stop_event = threading.Event()
    
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
    
        try:
            futures = []
    
            for (system_prompt, attempt_index, attempt_temp) in tasks:
                future = executor.submit(
                    self._process_attempt, 
                    user_input, 
                    system_prompt, 
                    attempt_index, 
                    stop_event, 
                    deadline,
                    attempt_temp
                )
    
                futures.append(future)
    
            # Collect ALL results — don't break on early stop.
            # Early stop signals remaining attempts to exit quickly,
            # but we still gather everything for a robust vote.
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
    
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
    
                    if not stop_event.is_set():
                        counts = Counter(valid_answers).most_common(1)
    
                        if counts and counts[0][1] >= self.cfg.early_stop:
                            stop_event.set()
    
                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue
    
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
    
        # --- Retry on all-None: if no valid answers and time remains, retry ---
        if not valid_answers and time.time() < deadline - 30:
            print('  >> All attempts returned None — retrying with remaining budget...')
            retry_stop = threading.Event()
            retry_executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
            try:
                retry_futures = []
                for retry_idx in range(self.cfg.attempts):
                    temp = self.cfg.temperature
                    # Use higher temperatures on retry to diversify
                    retry_temp = min(temp + 0.2, 1.0)
                    f = retry_executor.submit(
                        self._process_attempt,
                        user_input,
                        self.cfg.system_prompt,
                        retry_idx + self.cfg.attempts,
                        retry_stop,
                        deadline,
                        retry_temp
                    )
                    retry_futures.append(f)
                for future in as_completed(retry_futures):
                    try:
                        result = future.result()
                        detailed_results.append(result)
                        if result['Answer'] is not None:
                            valid_answers.append(result['Answer'])
                        if not retry_stop.is_set():
                            counts = Counter(valid_answers).most_common(1)
                            if counts and counts[0][1] >= self.cfg.early_stop:
                                retry_stop.set()
                    except Exception as exc:
                        print(f'Retry future failed: {exc}')
            finally:
                retry_stop.set()
                retry_executor.shutdown(wait=True, cancel_futures=True)

        self.problems_remaining = max(0, self.problems_remaining - 1)
    
        self._last_detailed_results = detailed_results
    
        if detailed_results:
            display_cols = ['Attempt', 'Answer', 'Python Calls', 'Python Errors', 'Response Length', 'Entropy', 'Time']
            results_dataframe = pd.DataFrame(detailed_results)
            display_df = results_dataframe[[c for c in display_cols if c in results_dataframe.columns]].copy()
            display_df['Entropy'] = display_df['Entropy'].round(3)
            display_df['Answer'] = display_df['Answer'].astype('Int64')
            
            display(display_df)
    
        if not valid_answers:
            print('\nResult: 0\n')
    
            return 0
    
        return self._select_answer(detailed_results)
    
    def __del__(self):
    
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
    
        if hasattr(self, 'log_file'):
            self.log_file.close()
    
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
    
                except Exception:
                    pass

In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
    id_value = id_.item(0)
    question_text = question.item(0)
    
    gc.disable()
    
    final_answer = solver.solve_problem(question_text)
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()

else:
    # Find test.csv — path varies between test runs and competition reruns
    test_csv_candidates = [
        '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',
        '/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv',
    ]
    test_csv = next((p for p in test_csv_candidates if os.path.isfile(p)), test_csv_candidates[0])
    print(f'Using test CSV: {test_csv}')

    inference_server.run_local_gateway(
        (test_csv,)
    )

In [ ]:
# =============================================================================
# ENHANCED TEST FRAMEWORK — Maximum Diagnostic Logging
# =============================================================================
# Diagnostic output is written to /kaggle/working/diagnostic.log (persists as kernel output)
# This avoids Kaggle's stdout truncation and UI hanging on large logs.

import sys

class TeeLogger:
    """Write to both stdout and a log file. Survives Kaggle log truncation."""
    def __init__(self, filepath='/kaggle/working/diagnostic.log'):
        self.terminal = sys.stdout
        self.log = open(filepath, 'w', buffering=1)  # line-buffered for live output
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
    def flush(self):
        self.terminal.flush()
        self.log.flush()

sys.stdout = TeeLogger()
sys.stderr = TeeLogger('/kaggle/working/diagnostic_stderr.log')

# =============================================================================
# v23 Test Framework — 3 Tiers
#   Tier 0: Priority Debug (2 problems x2 = 4 slots) — quick trial
#   Tier 1: At-Risk (6 problems)
#   Tier 2: Val Bench (loaded from CSV, ~70 new unseen problems)
#
# TEST_TIER controls how many tiers run:
#   0 = Tier 0 only (4 problem slots, ~5 min) — quick trial
#   1 = Tier 0 + 1 (10 problem slots, ~15 min)
#   2 = All tiers (80+ problems, full run)
# =============================================================================

TEST_TIER = 2  # Change to 0 for quick trial, 1 for medium, 2 for full

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):

    def _print_attempt_full(r, expected):
        """Print complete diagnostic for one attempt — every step, no truncation."""
        attempt = r.get('Attempt', '?')
        answer = r.get('Answer', None)
        entropy = r.get('Entropy', float('inf'))
        py_calls = r.get('Python Calls', 0)
        py_errors = r.get('Python Errors', 0)
        resp_len = r.get('Response Length', 0)
        temp = r.get('Temperature', '?')
        elapsed = r.get('Time', '?')
        status = 'CORRECT' if answer == expected else ('WRONG' if answer is not None else 'NONE')
        print(f'\n    --- Attempt {attempt} [{status}] answer={answer} expected={expected} temp={temp} entropy={entropy:.3f} ---')
        print(f'        python_calls={py_calls} errors={py_errors} tokens={resp_len} time={elapsed}s')

        conv = r.get('Conversation', [])
        if conv:
            imports_found = set()
            for entry in conv:
                if entry.get('type') == 'code_call':
                    code = entry.get('code', '')
                    for line in code.split('\n'):
                        line = line.strip()
                        if line.startswith('import ') or line.startswith('from '):
                            imports_found.add(line)
            if imports_found:
                print(f'        Libraries: {", ".join(sorted(imports_found))}')

            for entry in conv:
                turn = entry.get('turn', '?')
                etype = entry.get('type', '?')
                if etype == 'reasoning':
                    text = entry.get('text', '')
                    print(f'\n        [Turn {turn} REASONING] ({len(text)} chars)')
                    print(f'        {text}')
                elif etype == 'code_call':
                    code = entry.get('code', '')
                    output = entry.get('output', '')
                    is_error = entry.get('error', False)
                    err_marker = ' [ERROR]' if is_error else ''
                    print(f'\n        [Turn {turn} CODE{err_marker}]')
                    print(f'        >>> Code:')
                    for cline in code.split('\n'):
                        print(f'            {cline}')
                    print(f'        >>> Output:')
                    for oline in output.split('\n')[:50]:
                        print(f'            {oline}')
                    if len(output.split('\n')) > 50:
                        print(f'            ... ({len(output.split(chr(10)))} lines total)')

    def _get_gpu_stats():
        """Get GPU memory stats if available."""
        try:
            import torch
            if torch.cuda.is_available():
                alloc = torch.cuda.memory_allocated() / 1e9
                reserved = torch.cuda.memory_reserved() / 1e9
                return f'GPU mem: {alloc:.1f}GB alloc / {reserved:.1f}GB reserved'
        except Exception:
            pass
        return 'GPU stats unavailable'

    def run_test_batch(name, problems, solver):
        """Run a batch of test problems with full diagnostics."""
        print(f'\n{"="*80}')
        print(f'  BATCH: {name} ({len(problems)} problems)')
        print(f'  {_get_gpu_stats()}')
        print(f'{"="*80}')

        batch_correct = 0
        batch_total = len(problems)
        batch_results = []
        batch_start = time.time()

        for idx, p in enumerate(problems):
            pid = p['id']
            expected = p['answer']
            problem_text = p['problem']

            print(f'\n{"─"*70}')
            print(f'  [{idx+1}/{batch_total}] Problem {pid} | Expected: {expected}')
            print(f'{"─"*70}')

            prob_start = time.time()
            predicted = solver.solve_problem(problem_text)
            prob_elapsed = time.time() - prob_start

            correct = (predicted == expected)
            batch_correct += int(correct)
            status = 'CORRECT' if correct else 'WRONG'

            result_entry = {
                'id': pid,
                'predicted': predicted,
                'expected': expected,
                'correct': correct,
                'time': round(prob_elapsed, 1),
            }

            detailed = getattr(solver, '_last_detailed_results', [])
            if detailed:
                answers = [r['Answer'] for r in detailed if r.get('Answer') is not None]
                from collections import Counter
                vote_dist = dict(Counter(answers).most_common())
                result_entry['votes'] = vote_dist
                result_entry['num_answers'] = len(answers)
                result_entry['num_none'] = sum(1 for r in detailed if r.get('Answer') is None)
                result_entry['num_errors'] = sum(r.get('Python Errors', 0) for r in detailed)
                es_hit = any(count >= CFG.early_stop for count in Counter(answers).values()) if answers else False
                result_entry['early_stopped'] = es_hit

                print(f'\n  STATUS: {status} | Predicted: {predicted} | Expected: {expected} | Time: {prob_elapsed:.1f}s')
                print(f'  Votes: {vote_dist} | Answers: {len(answers)} | Nones: {result_entry["num_none"]} | Errors: {result_entry["num_errors"]} | ES: {es_hit}')

                if not correct:
                    print(f'\n  WRONG ANSWER ANALYSIS:')
                    for r in detailed:
                        if r.get('Answer') is not None and r['Answer'] != expected:
                            print(f'    Attempt {r["Attempt"]}: answered {r["Answer"]} (entropy={r["Entropy"]:.3f})')
                    correct_attempts = [r for r in detailed if r.get('Answer') == expected]
                    if correct_attempts:
                        print(f'    {len(correct_attempts)} attempt(s) got the right answer but were outvoted')

                for r in detailed:
                    _print_attempt_full(r, expected)
            else:
                print(f'\n  STATUS: {status} | Predicted: {predicted} | Expected: {expected} | Time: {prob_elapsed:.1f}s')

            batch_results.append(result_entry)

            running_pct = batch_correct / (idx + 1) * 100
            print(f'\n  Running: {batch_correct}/{idx+1} ({running_pct:.0f}%) | Elapsed: {time.time()-batch_start:.0f}s')

        batch_elapsed = time.time() - batch_start
        print(f'\n{"="*80}')
        print(f'  BATCH COMPLETE: {name}')
        print(f'  Score: {batch_correct}/{batch_total} ({batch_correct/max(batch_total,1)*100:.0f}%)')
        print(f'  Time: {batch_elapsed:.0f}s')
        if batch_results:
            times = [r['time'] for r in batch_results]
            print(f'  Timing: min={min(times):.0f}s  max={max(times):.0f}s  median={sorted(times)[len(times)//2]:.0f}s')
        print(f'{"="*80}')

        return batch_correct, batch_total, batch_results

    def load_csv_problems(problems_csv, answers_csv):
        """Load problems and answers from CSV files."""
        import csv
        problems = {}
        answers = {}
        with open(problems_csv, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                problems[row['id']] = row['problem']
        with open(answers_csv, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                answers[row['id']] = int(row['answer'])
        return [{'id': pid, 'problem': problems[pid], 'answer': answers[pid]}
                for pid in problems if pid in answers]

    # =========================================================================
    #  TIER 0: Priority Debug (2 problems x 2 runs = 4 problem slots)
    #  These are solved TWICE each to check consistency.
    # =========================================================================
    PRIORITY_PROBLEMS = [
        {'id': '86e8e5', 'problem': 'Let $n \\geq 6$ be a positive integer. We call a positive integer $n$-Norwegian if it has three distinct positive divisors whose sum is equal to $n$. Let $f(n)$ denote the smallest $n$-Norwegian positive integer. Let $M=3^{2025!}$ and for a non-negative integer $c$ define $g(c)=\\frac{1}{2025!}\\left\\lfloor \\frac{2025! f(M+c)}{M}\\right\\rfloor.$ We can write $g(0)+g(4M)+g(1848374)+g(10162574)+g(265710644)+g(44636594)=\\frac{p}{q}$ where $p$ and $q$ are coprime positive integers. What is the remainder when $p+q$ is divided by $99991$?', 'answer': 8687},
        {'id': '86e8e5', 'problem': 'Let $n \\geq 6$ be a positive integer. We call a positive integer $n$-Norwegian if it has three distinct positive divisors whose sum is equal to $n$. Let $f(n)$ denote the smallest $n$-Norwegian positive integer. Let $M=3^{2025!}$ and for a non-negative integer $c$ define $g(c)=\\frac{1}{2025!}\\left\\lfloor \\frac{2025! f(M+c)}{M}\\right\\rfloor.$ We can write $g(0)+g(4M)+g(1848374)+g(10162574)+g(265710644)+g(44636594)=\\frac{p}{q}$ where $p$ and $q$ are coprime positive integers. What is the remainder when $p+q$ is divided by $99991$?', 'answer': 8687},
        {'id': '76aef9', 'problem': 'Players $A$ and $B$ play a game on a blackboard that initially contains 1997 copies of the number 1. In every round, player $A$ erases two numbers $x$ and $y$ from the blackboard, and then player $B$ writes one of the numbers $x+y$ and $|x-y|$ on the blackboard. Additionally, there is a coin that is flipped before every round. If the coin lands heads, player $B$ is forced to write $x+y$ and if the coin lands tails player $B$ is forced to write $|x-y|$. The game terminates as soon as, at the end of some round, one of the following holds:\n\n(1) one of the numbers on the blackboard is larger than the sum of all other numbers;\n\n(2) there are only zeros on the blackboard.\n\nPlayer $B$ must then give as many cookies to player $A$ as there are numbers on the blackboard. Player $A$ wants to get as many cookies as possible, whereas player $B$ wants to give as few as possible. Determine the number of cookies that $A$ receives if both players play optimally.', 'answer': 8},
        {'id': '76aef9', 'problem': 'Players $A$ and $B$ play a game on a blackboard that initially contains 1997 copies of the number 1. In every round, player $A$ erases two numbers $x$ and $y$ from the blackboard, and then player $B$ writes one of the numbers $x+y$ and $|x-y|$ on the blackboard. Additionally, there is a coin that is flipped before every round. If the coin lands heads, player $B$ is forced to write $x+y$ and if the coin lands tails player $B$ is forced to write $|x-y|$. The game terminates as soon as, at the end of some round, one of the following holds:\n\n(1) one of the numbers on the blackboard is larger than the sum of all other numbers;\n\n(2) there are only zeros on the blackboard.\n\nPlayer $B$ must then give as many cookies to player $A$ as there are numbers on the blackboard. Player $A$ wants to get as many cookies as possible, whereas player $B$ wants to give as few as possible. Determine the number of cookies that $A$ receives if both players play optimally.', 'answer': 8},
    ]

    # =========================================================================
    #  TIER 1: At-Risk Problems (6 problems that showed instability in v22)
    # =========================================================================
    AT_RISK_PROBLEMS = [
        {'id': '424e18', 'problem': 'A tournament is held with $2^{20}$ runners each of which has a different running speed. In each race, two runners compete against each other with the faster runner always winning the race. The competition consists of $20$ rounds with each runner starting with a score of $0$. In each round, the runners are paired in such a way that in each pair, both runners have the same score at the beginning of the round. The winner of each race in the $i^{\\text{th}}$ round receives $2^{20-i}$ points and the loser gets no points. At the end of the tournament, we rank the competitors according to their scores. Let $N$ denote the number of possible orderings of the competitors at the end of the tournament. Let $k$ be the largest positive integer such that $10^k$ divides $N$. What is the remainder when $k$ is divided by $10^{5}$?', 'answer': 21818},
        {'id': 'dd7f5e', 'problem': 'Let $\\mathcal{F}$ be the set of functions $\\alpha \\colon \\mathbb{Z}\\to \\mathbb{Z}$ for which there are only finitely many $n \\in \\mathbb{Z}$ such that $\\alpha(n) \\neq 0$. For two functions $\\alpha$ and $\\beta$ in $\\mathcal{F}$, define their product $\\alpha\\star\\beta$ to be $\\sum\\limits_{n\\in\\mathbb{Z}} \\alpha(n)\\cdot \\beta(n)$. Also, for $n\\in\\mathbb{Z}$, define a shift operator $S_n \\colon \\mathcal{F}\\to \\mathcal{F}$ by $S_n(\\alpha)(t)=\\alpha(t+n)$ for all $t \\in \\mathbb{Z}$. A function $\\alpha \\in \\mathcal{F}$ is called shifty if $\\alpha(m)=0$ for all integers $m<0$ and $m>8$ and there exists $\\beta \\in \\mathcal{F}$ and integers $k \\neq l$ such that for all $n \\in \\mathbb{Z}$, $S_n(\\alpha)\\star\\beta = 1$ if $n \\in \\{k,l\\}$ and $0$ if $n \\not\\in \\{k,l\\}$. How many shifty functions are there in $\\mathcal{F}$?', 'answer': 160},
        {'id': 'b4ec47', 'problem': 'Find the number of rectangles that can be formed inside a fixed regular dodecagon ( $12$ -gon) where each side of the rectangle lies on either a side or a diagonal of the dodecagon. The diagram below shows three of those rectangles. [asy] unitsize(0.6 inch); for(int i=0; i<360; i+=30) { dot(dir(i), 4+black); draw(dir(i)--dir(i+30)); } draw(dir(120)--dir(330)); filldraw(dir(210)--dir(240)--dir(30)--dir(60)--cycle, mediumgray, linewidth(1.5)); draw((0,0.366)--(0.366,0), linewidth(1.5)); [/asy]', 'answer': 315},
        {'id': '2d282e', 'problem': 'A teacher wants to divide a class of $d$ students into $k$ groups (some of which may be empty). Each student $i$ has a score $a_i$ between 0 and 1, inclusive, and the sum of the scores of all students is $59$. The teacher wants to partition the students into groups such that the sum of the scores in each group is at most 1. Find the smallest integer $k$ such that this is always possible, regardless of the number of students $d$ and their scores $a_1, \\ldots, a_d$.', 'answer': 117},
        {'id': '8fea51', 'problem': 'Let $b_m b_{m-1}\\cdots b_0$ be the base-7 representation of a positive integer $n$ for some positive integer $m$. Let $a_i$ be the number obtained by removing the digit $b_i$ from the base-7 representation of $m$ (read in base 7). Find the number of $n$ that satisfy $n=\\sum_{i=0}^{m-1}a_i$.', 'answer': 42},
        {'id': '269012', 'problem': 'A cube-shaped container has vertices $A,$ $B,$ $C,$ and $D,$ where $\\overline{AB}$ and $\\overline{CD}$ are parallel edges of the cube, and $\\overline{AC}$ and $\\overline{BD}$ are diagonals of faces of the cube, as shown. Vertex $A$ of the cube is set on a horizontal plane $\\mathcal{P}$ so that the plane of the rectangle $ABDC$ is perpendicular to $\\mathcal{P},$ vertex $B$ is $2$ meters above $\\mathcal{P},$ vertex $C$ is $8$ meters above $\\mathcal{P},$ and vertex $D$ is $10$ meters above $\\mathcal{P}.$ The cube contains water whose surface is parallel to $\\mathcal{P}$ at a height of $7$ meters above $\\mathcal{P}.$ The volume of water is $\\frac{m}{n}$ cubic meters, where $m$ and $n$ are relatively prime positive integers. Find $m+n.$ [asy] //Made by Djmathman (orz) size(250); defaultpen(linewidth(0.6)); pair A = origin, B = (6,3), X = rotate(40)*B, Y = rotate(70)*X, C = X+Y, Z = X+B, D = B+C, W = B+Y; pair P1 = 0.8*C+0.2*Y, P2 = 2/3*C+1/3*X, P3 = 0.2*D+0.8*Z, P4 = 0.63*D+0.37*W; pair E = (-20,6), F = (-6,-5), G = (18,-2), H = (9,8); filldraw(E--F--G--H--cycle,rgb(0.98,0.98,0.2)); fill(A--Y--P1--P4--P3--Z--B--cycle,rgb(0.35,0.7,0.9)); draw(A--B--Z--X--A--Y--C--X^^C--D--Z); draw(P1--P2--P3--P4--cycle^^D--P4); dot("$A$",A,S); dot("$B$",B,S); dot("$C$",C,N); dot("$D$",D,N); label("$\\mathcal P$",(-13,4.5)); [/asy]', 'answer': 751},
    ]

    # =========================================================================
    #  RUN TIERS (controlled by TEST_TIER)
    # =========================================================================
    all_results = []
    total_correct = 0
    total_problems = 0
    suite_start = time.time()

    # --- TIER 0: Priority Debug (dual-run, 4 problem slots) ---
    c, t, r = run_test_batch('PRIORITY DEBUG (dual-run)', PRIORITY_PROBLEMS, solver)
    total_correct += c
    total_problems += t
    all_results.extend(r)

    # --- TIER 1: At-Risk Problems (6 problems) ---
    if TEST_TIER >= 1:
        c, t, r = run_test_batch('AT-RISK PROBLEMS', AT_RISK_PROBLEMS, solver)
        total_correct += c
        total_problems += t
        all_results.extend(r)
    else:
        print(f'\nSKIPPING Tier 1 (TEST_TIER={TEST_TIER})')

    # --- TIER 2: Val Bench (loaded from CSV, ~70 new unseen problems) ---
    if TEST_TIER >= 2:
        val_problems_csv = '/kaggle/input/aimo3-test-data/test_problems.csv'
        val_answers_csv = '/kaggle/input/aimo3-test-data/test_answers.csv'
        if os.path.isfile(val_problems_csv) and os.path.isfile(val_answers_csv):
            val_problems = load_csv_problems(val_problems_csv, val_answers_csv)
            c, t, r = run_test_batch('VAL BENCH (new unseen)', val_problems, solver)
            total_correct += c
            total_problems += t
            all_results.extend(r)
        else:
            print(f'SKIPPING Tier 2: CSV not found at {val_problems_csv}')
    else:
        print(f'\nSKIPPING Tier 2 (TEST_TIER={TEST_TIER})')

    # =========================================================================
    #  DOUBLE-RUN RETRY: Re-run failed problems to simulate competition scoring
    # =========================================================================
    wrong = [r for r in all_results if not r['correct']]
    if wrong:
        print(f'\n{"="*80}')
        print(f'  DOUBLE-RUN RETRY: {len(wrong)} failed problems')
        print(f'{"="*80}')

        # Build retry list from all tiers
        retry_problems = []
        for p in PRIORITY_PROBLEMS + AT_RISK_PROBLEMS:
            if any(r['id'] == p['id'] and not r['correct'] for r in all_results):
                if not any(rp['id'] == p['id'] and rp['problem'] == p['problem'] for rp in retry_problems):
                    retry_problems.append(p)

        if retry_problems:
            retry_c, retry_t, retry_results = run_test_batch(
                'DOUBLE-RUN RETRY', retry_problems, solver
            )

            print(f'\n  DOUBLE-RUN SCORING SIMULATION:')
            for p in retry_problems:
                first_results = [r for r in all_results if r['id'] == p['id']]
                retry_r = [r for r in retry_results if r['id'] == p['id']]
                first_correct = any(r['correct'] for r in first_results)
                retry_correct = any(r['correct'] for r in retry_r)
                if first_correct and retry_correct:
                    score = 1.0
                elif first_correct or retry_correct:
                    score = 0.5
                else:
                    score = 0.0
                print(f'    {p["id"]}: run1={first_correct} run2={retry_correct} → {score} pts')

    # =========================================================================
    #  GPU METRICS SUMMARY
    # =========================================================================
    try:
        solver.print_gpu_summary()
    except Exception as e:
        print(f'GPU metrics unavailable: {e}')

    # =========================================================================
    #  FINAL SUMMARY
    # =========================================================================
    suite_elapsed = time.time() - suite_start
    print(f'\n{"="*80}')
    print(f'  FINAL SUMMARY')
    print(f'{"="*80}')
    print(f'  Score: {total_correct}/{total_problems} ({total_correct/max(total_problems,1)*100:.1f}%)')
    print(f'  Total time: {suite_elapsed:.0f}s ({suite_elapsed/60:.1f} min)')
    print(f'  TEST_TIER: {TEST_TIER}')

    if all_results:
        print(f'\n  {"ID":<8} {"Status":<8} {"Pred":>8} {"Exp":>8} {"Votes":<24} {"Ans":<6} {"Errs":<5} {"Time":>6} {"ES":<3}')
        print(f'  {"─"*8} {"─"*8} {"─":>8} {"─":>8} {"─"*24} {"─"*6} {"─"*5} {"─":>6} {"─"*3}')
        for r in all_results:
            status = 'OK' if r['correct'] else 'FAIL'
            votes_str = str(r.get('votes', {}))
            if len(votes_str) > 22:
                votes_str = votes_str[:22] + '..'
            n_ans = r.get('num_answers', '?')
            n_err = r.get('num_errors', '?')
            es = 'Y' if r.get('early_stopped', False) else 'N'
            print(f'  {r["id"]:<8} {status:<8} {r["predicted"]:>8} {r["expected"]:>8} {votes_str:<24} {str(n_ans):<6} {str(n_err):<5} {r["time"]:>5.0f}s {es:<3}')

        wrong_results = [r for r in all_results if not r['correct']]
        correct_results = [r for r in all_results if r['correct']]
        if wrong_results:
            print(f'\n  WRONG ANSWERS ({len(wrong_results)}):')
            for r in wrong_results:
                print(f'    {r["id"]}: predicted {r["predicted"]}, expected {r["expected"]}, votes={r.get("votes", "?")}')
        if correct_results:
            avg_time = sum(r['time'] for r in correct_results) / len(correct_results)
            print(f'\n  Correct answers avg time: {avg_time:.0f}s')

    print(f'\n{"="*80}')
